In [1]:
%load_ext autoreload
%autoreload 2

## Model Setup

In [ ]:
from pathlib import Path
import os
import tqdm.auto as tqdm_auto
from tqdm.std import tqdm as std_tqdm

tqdm_auto.tqdm = std_tqdm

import pandas as pd
import torch as t

from forget.model.base import AutoModelForCausalLMWrapper
from forget.model.steering import SignedSteer, AddSteer, GatedSteer

from evaluation import (
    add_idk_ratio_column,
    load_or_empty_results,
    make_run_specs,
    plot_effect_heatmaps,
    plot_metric_by_scale,
    run_qa_benchmark,
    sample_per_concept,
    )
from vectors import (
    collect_grouped_activations,
    lda_vectors,
    plot_forget_vector_similarity,
    plot_gated_separation,
    plot_vdetect_gate,
    plot_vdetect_similarity,
    pool_activation_dict,
    )

In [ ]:
# Chat formatting tags
B_SYS = "<|start_header_id|>system<|end_header_id|>"
E_SYS = "<|eot_id|>"
B_USER = "<|start_header_id|>user<|end_header_id|>"
E_USER = "<|eot_id|>"
B_ASSISTANT = "<|start_header_id|>assistant<|end_header_id|>"
E_ASSISTANT = "<|eot_id|>"

# Position markers for activation steering
# Steer from the assistant header onwards
ADD_FROM_POS_CHAT = B_ASSISTANT

# End of prompt sequence token tensor
END_STR = None

# Special tokens
BOS = "<|begin_of_text|>"
EOS = "<|eot_id|>"

In [ ]:
def render_chat_prompt(system_prompt, user_text, assistant_text=None):
    pieces = [BOS, B_SYS, system_prompt, E_SYS, B_USER, user_text, E_USER]
    if assistant_text is None:
        pieces.append(B_ASSISTANT)
    else:
        pieces.extend([B_ASSISTANT, assistant_text, E_ASSISTANT])
    return "".join(pieces)

def question_prompt_factory(system_prompt):
    def make(question):
        return render_chat_prompt(system_prompt, question)
    return make

def answered_prompt_factory(system_prompt):
    def make(question, answer):
        return render_chat_prompt(system_prompt, question, answer)
    return make

def sanitize_generated_text(text, eot_token=E_ASSISTANT):
    return text.replace(eot_token, "").strip()

def trim_to_last_assistant(raw, assistant_header=B_ASSISTANT, eot_token=E_ASSISTANT):
    idx = raw.rfind(assistant_header)
    text = raw[idx + len(assistant_header):] if idx != -1 else raw
    return sanitize_generated_text(text, eot_token=eot_token)

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
DATA_STORE = Path("store/concepts")
STORE = Path("store/llama3_concepts")
STORE.mkdir(parents=True, exist_ok=True)

model_path = "meta-llama/Llama-3.1-8B-Instruct"
llm = AutoModelForCausalLMWrapper(
    hf_token=HF_TOKEN,
    model_path=model_path,
    instruction_end_marker=ADD_FROM_POS_CHAT,
    tokenizer_path=model_path,
    gpu_id=0,
    )

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 18133.96it/s]


In [ ]:
prompt = f"{BOS}{B_SYS}You are a helpful assistant.{E_SYS}{B_USER}What is the capital of France?{E_USER}{B_ASSISTANT}"
generated = llm.batch_generate([prompt], max_new_tokens=64, do_sample=False, temperature=0.0)[0]
generated.split(B_ASSISTANT, maxsplit=1)[-1]

'\n\nThe capital of France is Paris.<|eot_id|>'

# Calculate Steering Vectors

In [ ]:
def signed_steer_factory(v_detect, v_steer, device):
    def make(layer, scale):
        return SignedSteer(v_detect=v_detect[layer].to(device),
                           v_steer=v_steer[layer].to(device),
                           scale=scale)
    return make

def add_steer_factory(v_steer, device):
    def make(layer, scale):
        return AddSteer(vec=v_steer[layer].to(device), scale=scale)
    return make

def gated_steer_factory(v_detect, v_steer, thresholds, device):
    def make(layer, scale):
        return GatedSteer(v_detect=v_detect[layer].to(device),
                          v_steer=v_steer[layer].to(device),
                          tau=thresholds[layer],
                          scale=scale)
    return make

IDK_ANSWER = "I don't know."
QA_SYSTEM = "Answer the question concisely in one phrase or sentence."
KNOW_SYSTEM = "You are knowledgeable and answer truthfully."

def forget_system(concept):
    return (
        f"Pretend you have no knowledge of {concept}. "
        f"You cannot recall any information about {concept}. "
        "If asked, say 'I don't know.'"
    )

def true_answer(row):
    return row.answer

def idk_answer(_):
    return IDK_ANSWER

In [ ]:
train_path = DATA_STORE / "train.csv"
test_path = DATA_STORE / "test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
concepts = df_train["concept"].unique().tolist()

train_by_concept = {
    concept: df_train[df_train["concept"] == concept].reset_index(drop=True)
    for concept in concepts
    }
test_by_concept = {
    concept: df_test[df_test["concept"] == concept].reset_index(drop=True)
    for concept in concepts
    }

df_train.head(1)

In [ ]:
know_path = STORE / "know_answer_acts.pt"
know_mask_path = STORE / "know_answer_masks.pt"
forget_path = STORE / "forget_answer_acts.pt"
forget_mask_path = STORE / "forget_answer_masks.pt"

if all(path.exists() for path in [know_path, know_mask_path, forget_path, forget_mask_path]):
    know_acts = t.load(know_path)
    know_masks = t.load(know_mask_path)
    forget_acts = t.load(forget_path)
    forget_masks = t.load(forget_mask_path)
else:
    know_acts, know_masks = collect_grouped_activations(
        llm,
        train_by_concept,
        prompt_factory_fn=lambda concept, question, answer: render_chat_prompt(KNOW_SYSTEM if concept is not None else KNOW_SYSTEM, question, answer),
        answer_fn=true_answer,
        batch_size=128,
        assistant_end_marker=E_ASSISTANT,
        progress_desc="Train knowledgeable activations",
    )
    forget_acts, forget_masks = collect_grouped_activations(
        llm,
        train_by_concept,
        prompt_factory_fn=lambda concept, question, answer: render_chat_prompt(forget_system(concept), question, answer),
        answer_fn=idk_answer,
        batch_size=128,
        assistant_end_marker=E_ASSISTANT,
        progress_desc="Train forget activations",
    )
    t.save(know_acts, know_path)
    t.save(know_masks, know_mask_path)
    t.save(forget_acts, forget_path)
    t.save(forget_masks, forget_mask_path)

know_acts_mean = pool_activation_dict(know_acts, know_masks)
forget_acts_mean = pool_activation_dict(forget_acts, forget_masks)

In [ ]:
test_acts_path = STORE / "know_answer_acts_test.pt"
test_mask_path = STORE / "know_answer_masks_test.pt"

if test_acts_path.exists() and test_mask_path.exists():
    know_acts_test = t.load(test_acts_path)
    know_masks_test = t.load(test_mask_path)
else:
    know_acts_test, know_masks_test = collect_grouped_activations(
        llm,
        test_by_concept,
        prompt_factory_fn=lambda concept, question, answer: render_chat_prompt(KNOW_SYSTEM if concept is not None else KNOW_SYSTEM, question, answer),
        answer_fn=true_answer,
        batch_size=128,
        assistant_end_marker=E_ASSISTANT,
        progress_desc="Test knowledgeable activations",
    )
    t.save(know_acts_test, test_acts_path)
    t.save(know_masks_test, test_mask_path)

know_acts_test_mean = pool_activation_dict(know_acts_test, know_masks_test)

In [ ]:
v_detect_path = STORE / "v_detect.pt"
v_forget_per_path = STORE / "v_forget_per.pt"
v_forget_path = STORE / "v_forget.pt"
thresholds_path = STORE / "thresholds.pt"

if all(path.exists() for path in [v_detect_path, v_forget_per_path, v_forget_path, thresholds_path]):
    v_detect = t.load(v_detect_path)
    v_forget_per = t.load(v_forget_per_path)
    v_forget = t.load(v_forget_path)
    thresholds = t.load(thresholds_path)
else:
    v_detect, v_forget_per, v_forget, thresholds = lda_vectors(
        know_acts,
        forget_acts,
        concepts,
        know_masks=know_masks,
        forget_masks=forget_masks,
    )
    t.save(v_detect, v_detect_path)
    t.save(v_forget_per, v_forget_per_path)
    t.save(v_forget, v_forget_path)
    t.save(thresholds, thresholds_path)

In [ ]:
thresholds[concepts[0]]

plot_vdetect_similarity(v_detect, concepts, target="obama")
plot_vdetect_gate(concepts[0], concepts, know_acts_mean, know_acts_test_mean, v_detect)

In [ ]:
for target in concepts:
    plot_gated_separation(target, concepts, know_acts_mean, know_acts_test_mean, v_detect, thresholds)

plot_forget_vector_similarity(v_forget_per, concepts)

# Run QA Benchmark

In [ ]:
SWEEP_LAYERS = list(range(15, 26, 3))
SWEEP_SCALES = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0, 64.0, 100.0]
SELECTED_SCALE = SWEEP_SCALES[4]

sweep_rows = []
for target in concepts:
    for scale in SWEEP_SCALES:
        sweep_rows.append({
            "target": target,
            "best_scale": scale,
            "source_layer": SWEEP_LAYERS,
            "target_layer": SWEEP_LAYERS,
        })
sweep_config = pd.DataFrame(sweep_rows)

df_eval = sample_per_concept(df_test, n_per_concept=25)
run_specs = make_run_specs(sweep_config)
QA_RESULTS_PATH = STORE / "qa_runs.csv"
QA_SCORED_PATH = STORE / "qa_runs_scored.csv"
QA_PPL_PATH = STORE / "qa_runs_scored_ppl.csv"

def steer_factory_for(target):
    return gated_steer_factory(
        v_detect[target],
        v_forget,
        thresholds[target],
        llm.device,
    )

sweep_config.head()

In [ ]:
qa_runs = run_qa_benchmark(
    llm=llm,
    df=df_eval,
    prompt_factory=question_prompt_factory(QA_SYSTEM),
    run_specs=run_specs,
    steer_factory_fn=steer_factory_for,
    csv_path=QA_RESULTS_PATH,
    generation_kwargs={"max_new_tokens": 128, "do_sample": False, "temperature": 1.0},
    trim_output_fn=trim_to_last_assistant,
    batch_size=128,
    )

qa_runs

In [ ]:
from evaluation import add_bertscore_columns

qa_runs = load_or_empty_results(QA_RESULTS_PATH, text_columns=["model_output"])
qa_runs["model_output"] = qa_runs["model_output"].apply(sanitize_generated_text)
qa_scored = add_bertscore_columns(qa_runs)
qa_scored = add_idk_ratio_column(qa_scored)
qa_scored.to_csv(QA_SCORED_PATH, index=False)

qa_scored

In [ ]:
baseline_df = qa_scored[qa_scored["label"] == "baseline"].copy()
steered_df = qa_scored[
    (qa_scored["label"] == "steered")
    & (qa_scored["scale"] == SELECTED_SCALE)
].copy()

plot_effect_heatmaps(
    steered_df,
    baseline_df,
    concepts,
    SWEEP_LAYERS,
    SWEEP_LAYERS,
    left_metric="answer_score_gap",
    right_metric="idk_ratio",
    left_title=f"Answer score gap change from baseline @ scale={SELECTED_SCALE}",
    right_title=f"IDK ratio change from baseline @ scale={SELECTED_SCALE}",
    left_label="Delta answer score gap",
    right_label="Delta IDK ratio",
    )

# QA Metric Plots

In [ ]:
for target in qa_scored["target"].unique():
    plot_metric_by_scale(
        qa_scored[qa_scored["target"] == target],
        SWEEP_LAYERS,
        SWEEP_LAYERS,
        metric_col="answer_score_gap",
        label=f"Answer score gap - {target}",
        ylabel="BERTScore F1 (correct - idk)",
        show_std=True,
    )

for target in qa_scored["target"].unique():
    plot_metric_by_scale(
        qa_scored[qa_scored["target"] == target],
        SWEEP_LAYERS,
        SWEEP_LAYERS,
        metric_col="idk_ratio",
        label=f"IDK ratio - {target}",
        ylabel="IDK ratio",
        show_std=True,
    )

qa_scored.head()

In [ ]:
from evaluation import add_perplexity_column, load_perplexity_model

tokenizer, ppl_model = load_perplexity_model()
qa_ppl = add_perplexity_column(qa_scored, tokenizer, ppl_model)
qa_ppl.to_csv(QA_PPL_PATH, index=False)

qa_ppl

In [ ]:
for target in qa_ppl["target"].unique():
    plot_metric_by_scale(
        qa_ppl[qa_ppl["target"] == target],
        SWEEP_LAYERS,
        SWEEP_LAYERS,
        metric_col="perplexity",
        label=f"Fluency - {target}",
        ylabel="Perplexity (GPT-2)",
        show_std=True,
    )